# Issue 1 — Dataset Sanity Check (Appendix A)

In [2]:
"""
=============================================================================
IoT-RPL 2021 — LEAKAGE AUDIT + FULL CLEANING PIPELINE (FINAL)
=============================================================================
Purpose:
  This script serves two roles:

  1. METHODOLOGICAL PROOF OF LEAKAGE
     Runs a full diagnostic on the raw IoT-RPL 2021 dataset before any
     cleaning, documenting exactly which features are leaky and why.
     This evidence supports the methodology section claim that the raw
     dataset cannot be used for valid attack-detection modeling without
     prior cleaning.

  2. CLEANING PIPELINE
     Removes leaky and unreadable columns, coerces remaining features to
     clean numerics, and produces a leakage-free dataset with a proper
     file-based train/test split.

Leakage diagnostic methodology (Appendix A pattern from feedback):
  Layer 1 — Per-class value range analysis
    - Constant range (min == max per class) = perfect class identifier
    - Fully disjoint ranges across classes  = structural leakage
    - KS uniformity test p > 0.05           = uniform distribution (suspicious)
  Layer 2 — Single-feature depth-4 decision tree (3-fold CV)
    - Accuracy > 0.50 + constant/disjoint   = RED FLAG (leakage)
    - Accuracy > 0.50 + overlapping ranges  = real signal (keep)
    - Chance level = 0.20 for 5 classes

Why each column group was removed:
  Direct identifiers : 'from', 'to'
    Node IDs that directly encode attack type — 'from' achieves acc = 1.0
  Protocol columns   : 'frame_proto', 'protocol'
    Constant = 0 for all attack classes; contain strings in Normal class
  Leaky flag columns : 'valid_lifetime', 'reserved',
                       'desti_prefix', 'desti_prefix.1', 'desti_prefix.2',
                       'desti_prefix.3'
    Constant = 0 for Blackhole, Rank, Version classes (perfect identifiers)
  Hex-string columns : 'DOAGID', 'DOAG_info', 'lifetime', 'prefix_info'
    Up to 57% of values are hex strings (e.g. '1E4000F1') — up to 99.3%
    zeros after coercion — astronomically large floats from hex misparse

Train/test split rationale:
  The same physical node appears in all 10 simulation files. Random CV
  folds would allow the model to see a node's behavioral fingerprint in
  training and recognise it in the test fold — a subtle form of leakage.
  File-based splitting (files 0-7 train, files 8-9 test) ensures the
  test set represents genuinely unseen simulation runs.

Outputs:
  iot_rpl_clean_full.csv  — full cleaned dataset (~10.2M rows, 8 features)
  iot_rpl_train.csv       — training set (files 0-7, ~8.2M rows)
  iot_rpl_test.csv        — test set    (files 8-9, ~2.0M rows)
  leakage_audit_report.txt — full before/after leakage report for paper

Final features (8 clean numeric columns):
  control_type, type_cont_messg, DOAGID.1, DOAGID.2,
  DOAGID.3, DIO_info, object_cont_pt, preferred_liftime
=============================================================================
"""

import pandas as pd
import numpy as np
from scipy import stats
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
import warnings
import time
import io

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

IOT_RPL_DIR   = "IoT-RPL Dataset 2021/"
IOT_RPL_FILES = 10
LABEL_COL     = "label"
FLOAT32_MAX   = 3.4e38

COL_NAMES = [
    'from', 'to', 'frame_proto', 'protocol', 'control_type', 'type_cont_messg',
    'DOAGID', 'DOAGID.1', 'DOAGID.2', 'DOAGID.3', 'DOAG_info', 'DIO_info',
    'object_cont_pt', 'lifetime', 'prefix_info', 'valid_lifetime',
    'preferred_liftime', 'reserved', 'desti_prefix', 'desti_prefix.1',
    'desti_prefix.2', 'desti_prefix.3', 'label'
]

# Columns that are direct node identifiers or structurally leaky
ID_LEAKY_COLS = [
    'from', 'to', 'frame_proto', 'protocol',
    'valid_lifetime', 'reserved',
    'desti_prefix', 'desti_prefix.1', 'desti_prefix.2', 'desti_prefix.3'
]

# Columns that are hex strings unreadable as numeric features
HEX_COLS = ['DOAGID', 'DOAG_info', 'lifetime', 'prefix_info']

# Audit log — everything printed also goes here for the report file
_audit_log = []

def log(msg=""):
    """Print and record every output line."""
    print(msg)
    _audit_log.append(msg)


# ============================================================
# STEP 1 — LOAD ALL 10 RAW FILES
# ============================================================

def step1_load():
    log("\n" + "=" * 70)
    log("STEP 1 — LOADING ALL 10 RAW FILES")
    log("=" * 70)

    t0     = time.time()
    chunks = []

    df0 = pd.read_csv(IOT_RPL_DIR + "0.csv", low_memory=False)
    df0['file_id'] = 0
    chunks.append(df0)
    log(f"  File 0 : {len(df0):,} packets  (header from file)")

    for i in range(1, IOT_RPL_FILES):
        df_i = pd.read_csv(
            IOT_RPL_DIR + f"{i}.csv",
            header=None, names=COL_NAMES, low_memory=False
        )
        df_i['file_id'] = i
        chunks.append(df_i)
        log(f"  File {i} : {len(df_i):,} packets")

    df = pd.concat(chunks, ignore_index=True)
    df = df.dropna(subset=[LABEL_COL])

    log(f"\n  Total packets  : {len(df):,}")
    log(f"  Total columns  : {len(df.columns)}")
    log(f"  Load time      : {time.time() - t0:.1f}s")
    log(f"\n  Label distribution (packet-level):")
    for cls, cnt in df[LABEL_COL].value_counts().items():
        log(f"    {str(cls).ljust(12)}: {cnt:,}")

    return df


# ============================================================
# SANITY CHECK — REUSABLE
# Returns: (red_flag_count, leaky_col_list, signal_col_list,
#           per_col_results dict for summary table)
# ============================================================

def run_sanity_check(df, label_col, feature_cols, title):
    log("\n" + "=" * 70)
    log(title)
    log("=" * 70)
    log(f"  Shape    : {df[feature_cols + [label_col]].shape}")
    log(f"  Features : {feature_cols}")
    log(f"  Classes  : {sorted(df[label_col].unique())}")

    red_flags    = 0
    leaky_cols   = []
    signal_cols  = []
    range_flags  = {}
    col_results  = {}   # for summary table

    # ----------------------------------------------------------
    # Layer 1 — Per-class value ranges + KS uniformity test
    # ----------------------------------------------------------
    log("\n--- Layer 1: Per-class value ranges & KS uniformity test ---")
    log("    Constant range (min==max)  -> structural leakage (strongest)")
    log("    Disjoint ranges            -> structural leakage")
    log("    KS p > 0.05                -> uniform distribution (suspicious)\n")

    for col in feature_cols:
        log(f"  {col}:")
        col_ranges  = {}
        col_red     = False
        col_issues  = []

        vals = pd.to_numeric(df[col], errors='coerce').fillna(0)
        vals = vals.clip(-FLOAT32_MAX, FLOAT32_MAX)

        for cls in sorted(df[label_col].unique()):
            sub = vals[df[label_col] == cls].values
            mn, mx = float(sub.min()), float(sub.max())
            rng = mx - mn
            col_ranges[cls] = (mn, mx)

            if rng == 0:
                flag = "  <- RED FLAG (constant)"
                red_flags += 1
                col_red = True
                col_issues.append(f"constant=0 for class {cls}")
                if col not in leaky_cols:
                    leaky_cols.append(col)
            else:
                ks_p = stats.kstest(
                    sub, 'uniform', args=(mn, rng + 1e-9)
                ).pvalue
                flag = "  <- RED FLAG (uniform)" if ks_p > 0.05 else ""
                if ks_p > 0.05:
                    red_flags += 1
                    col_issues.append(f"uniform dist for class {cls} (KS p={ks_p:.3f})")

            log(f"    {str(cls).ljust(12)}: min={mn:.4g}  max={mx:.4g}{flag}")

        # Disjoint ranges check
        sorted_r = sorted(col_ranges.values(), key=lambda x: x[0])
        if len(sorted_r) >= 2:
            disjoint = all(
                sorted_r[i][1] < sorted_r[i+1][0]
                for i in range(len(sorted_r) - 1)
            )
            if disjoint:
                log(f"    *** DISJOINT ranges across all classes "
                    f"<- STRUCTURAL LEAKAGE ***")
                red_flags += 1
                col_red = True
                col_issues.append("disjoint ranges across all classes")
                if col not in leaky_cols:
                    leaky_cols.append(col)

        range_flags[col] = col_red
        col_results[col] = {
            'layer1_red': col_red,
            'issues': col_issues,
            'ranges': col_ranges
        }

    # ----------------------------------------------------------
    # Layer 2 — Single-feature depth-4 accuracy (3-fold CV)
    # ----------------------------------------------------------
    log("\n--- Layer 2: Single-feature depth-4 accuracy (3-fold CV) ---")
    log("    Chance level = 0.20 for 5 classes")
    log("    > 0.50 + constant/disjoint = RED FLAG (leakage)")
    log("    > 0.50 + overlapping       = real signal (keep)\n")

    le = LabelEncoder()
    y  = le.fit_transform(df[label_col])

    for col in feature_cols:
        vals = (
            pd.to_numeric(df[col], errors='coerce')
            .fillna(0)
            .clip(-FLOAT32_MAX, FLOAT32_MAX)
            .values
            .astype(np.float32)
            .reshape(-1, 1)
        )

        score = cross_val_score(
            DecisionTreeClassifier(max_depth=4, random_state=42),
            vals, y, cv=3, n_jobs=-1
        ).mean()

        is_leaky = range_flags.get(col, False)
        is_high  = score > 0.50

        if is_high and is_leaky:
            tag = "  <- RED FLAG (leakage)"
            red_flags += 1
            if col not in leaky_cols:
                leaky_cols.append(col)
            col_results[col]['layer2_acc']  = score
            col_results[col]['layer2_flag'] = 'LEAKAGE'
        elif is_high and not is_leaky:
            tag = "  <- real signal (overlapping ranges, keep)"
            signal_cols.append(col)
            col_results[col]['layer2_acc']  = score
            col_results[col]['layer2_flag'] = 'SIGNAL'
        else:
            tag = ""
            col_results[col]['layer2_acc']  = score
            col_results[col]['layer2_flag'] = 'OK'

        log(f"  {col.ljust(22)}: acc = {score:.3f}{tag}")

    # ----------------------------------------------------------
    # Summary
    # ----------------------------------------------------------
    log("\n" + "=" * 70)
    log(f"  RESULT  : {red_flags} red flag(s)")
    log(f"  LEAKY   : {leaky_cols}")
    log(f"  SIGNAL  : {signal_cols}")
    log("=" * 70)

    return red_flags, leaky_cols, signal_cols, col_results


# ============================================================
# PHASE 0 — LEAKAGE AUDIT ON RAW DATA (proof for paper)
# ============================================================

def phase0_leakage_audit(df):
    log("\n" + "=" * 70)
    log("PHASE 0 — LEAKAGE AUDIT: RAW UNCLEANED DATASET")
    log("=" * 70)
    log("""
  This phase documents the leakage present in the raw IoT-RPL 2021
  dataset BEFORE any cleaning. The findings here constitute the
  methodological justification for the cleaning steps that follow.

  Red flags to watch for (per feedback Appendix A):
    (a) Any column with constant value per class
    (b) Any column with disjoint ranges across classes
    (c) Any single feature achieving > 50% depth-4 accuracy (chance = 20%)
    (d) KS uniformity p > 0.05 within a class (real data is never uniform)
""")

    # Run on ALL columns except file_id
    all_feature_cols = [c for c in df.columns if c not in [LABEL_COL, 'file_id']]

    flags, leaky, signal, col_results = run_sanity_check(
        df, LABEL_COL, all_feature_cols,
        "SANITY CHECK — RAW DATASET (BEFORE CLEANING)"
    )

    log(f"\n  AUDIT SUMMARY:")
    log(f"  Total red flags        : {flags}")
    log(f"  Leaky columns          : {len(leaky)}")
    log(f"  Columns with signal    : {len(signal)}")
    log(f"  Total features checked : {len(all_feature_cols)}")

    # Document the hand-coded rule equivalent (from feedback)
    log(f"\n  NOTE: The 'from' column (node ID) achieves acc=1.0 with a single")
    log(f"  depth-4 tree — equivalent to a direct label lookup by node identity.")
    log(f"  This is not attack detection; it is node recognition.")

    return flags, leaky, col_results


# ============================================================
# PHASE 1 — CLEANING
# ============================================================

def phase1_drop_bad_columns(df):
    log("\n" + "=" * 70)
    log("PHASE 1, STEP 1 — DROP IDENTIFIER + HEX COLUMNS")
    log("=" * 70)

    drop_id  = [c for c in ID_LEAKY_COLS if c in df.columns]
    drop_hex = [c for c in HEX_COLS      if c in df.columns]
    to_drop  = drop_id + drop_hex

    log(f"  Identifier/leaky cols dropped ({len(drop_id)}):")
    for c in drop_id:
        log(f"    {c}")
    log(f"  Hex-string cols dropped ({len(drop_hex)}):")
    for c in drop_hex:
        log(f"    {c}")

    df_out = df.drop(columns=to_drop)
    log(f"\n  Shape after drop : {df_out.shape}")
    log(f"  Remaining cols   : {[c for c in df_out.columns if c != 'file_id']}")
    return df_out


def phase1_coerce_and_clean(df):
    log("\n" + "=" * 70)
    log("PHASE 1, STEP 2 — COERCE TO NUMERIC, CAP OVERFLOW, FILL NaN")
    log("=" * 70)

    feature_cols = [c for c in df.columns if c not in [LABEL_COL, 'file_id']]
    df = df.copy()

    for col in feature_cols:
        orig_dtype = df[col].dtype
        df[col]    = pd.to_numeric(df[col], errors='coerce')
        n_nan      = df[col].isna().sum()
        n_over     = (df[col].abs() > FLOAT32_MAX).sum()
        df[col]    = df[col].clip(-FLOAT32_MAX, FLOAT32_MAX).fillna(0)

        log(f"  {col.ljust(22)}: orig={str(orig_dtype).ljust(8)} | "
            f"NaN filled={n_nan:,}  | overflow capped={n_over:,}")

    return df


# ============================================================
# PHASE 2 — POST-CLEANING SANITY CHECKS
# ============================================================

def phase2_sanity_check_1(df):
    log("\n" + "=" * 70)
    log("PHASE 2 — SANITY CHECK #1: AFTER INITIAL CLEANING")
    log("=" * 70)

    feature_cols = [c for c in df.columns if c not in [LABEL_COL, 'file_id']]
    return run_sanity_check(
        df, LABEL_COL, feature_cols,
        "SANITY CHECK #1 — POST-CLEANING"
    )


def phase2_drop_remaining(df, leaky_cols):
    log("\n" + "=" * 70)
    log("PHASE 2 — DROP REMAINING LEAKY FEATURES (if any)")
    log("=" * 70)

    to_drop = [c for c in leaky_cols if c in df.columns]
    if to_drop:
        log(f"  Dropping: {to_drop}")
        df = df.drop(columns=to_drop)
        log(f"  Shape after drop: {df.shape}")
    else:
        log("  Nothing to drop — all remaining features are clean.")
    return df


def phase2_final_sanity_check(df):
    log("\n" + "=" * 70)
    log("PHASE 2 — SANITY CHECK #2: FINAL CONFIRMATION")
    log("=" * 70)

    feature_cols = [c for c in df.columns if c not in [LABEL_COL, 'file_id']]
    flags, _, signal, col_results = run_sanity_check(
        df, LABEL_COL, feature_cols,
        "SANITY CHECK #2 — FINAL CLEAN DATASET"
    )

    if flags == 0:
        log("\n  ✓ Dataset is clean — 0 red flags. Ready for modeling.")
    else:
        log(f"\n  ⚠  {flags} red flag(s) remain — review above before modeling.")

    return flags, col_results


# ============================================================
# BEFORE / AFTER SUMMARY REPORT
# ============================================================

def print_before_after_report(raw_flags, raw_col_results,
                               clean_flags, clean_col_results,
                               dropped_cols):
    log("\n" + "=" * 70)
    log("BEFORE / AFTER LEAKAGE AUDIT REPORT")
    log("(Suitable for inclusion in methodology section)")
    log("=" * 70)

    log(f"""
  SUMMARY
  -------
  Raw dataset red flags   : {raw_flags}
  Clean dataset red flags : {clean_flags}
  Columns removed         : {len(dropped_cols)}
  Columns retained        : {len(clean_col_results)}
""")

    log("  COLUMNS REMOVED AND WHY")
    log("  " + "-" * 60)

    removal_reasons = {
        'from'          : "Node ID — acc=1.0 single-feature (direct label lookup)",
        'to'            : "Destination ID — identifier column",
        'frame_proto'   : "Constant=0 for all attack classes; strings in Normal",
        'protocol'      : "Constant=0 for all attack classes; strings in Normal",
        'valid_lifetime': "Constant=0 for Blackhole, Rank, Version classes",
        'reserved'      : "Constant=0 for Blackhole, Rank, Version classes",
        'desti_prefix'  : "Constant=0 for Blackhole, Rank, Version classes",
        'desti_prefix.1': "Constant=0 for Blackhole, Rank, Version classes",
        'desti_prefix.2': "Constant=0 for Blackhole, Rank, Version classes",
        'desti_prefix.3': "Constant=0 for Blackhole, Rank, Version classes",
        'DOAGID'        : "57.7% hex strings (e.g. '1E4000F1'); 98.7% zeros after coerce",
        'DOAG_info'     : "Hex strings; extreme values up to 4.7e+104 from hex misparse",
        'lifetime'      : "49.6% hex strings; 99.3% zeros after coercion",
        'prefix_info'   : "12.4% hex strings; 91.3% zeros after coercion",
        'file_id'       : "Metadata column (simulation file index), not a feature",
    }

    for col in dropped_cols:
        reason = removal_reasons.get(col, "Flagged as leaky by sanity check")
        log(f"  {col.ljust(22)}: {reason}")

    log(f"\n  RETAINED FEATURES AND THEIR STATUS")
    log("  " + "-" * 60)
    log(f"  {'Feature'.ljust(22)}  {'Layer2 Acc'.ljust(12)}  Status")
    log(f"  {'-------'.ljust(22)}  {'----------'.ljust(12)}  ------")

    for col, res in clean_col_results.items():
        acc    = res.get('layer2_acc', 0)
        status = res.get('layer2_flag', 'OK')
        log(f"  {col.ljust(22)}  {str(round(acc, 3)).ljust(12)}  {status}")

    log(f"""
  INTERPRETATION
  --------------
  All retained features have overlapping value ranges across the five
  attack classes (Normal, Flooding, Blackhole, Rank, Version). No single
  feature achieves accuracy above 0.50 with a depth-4 decision tree,
  consistent with a chance level of 0.20 for a 5-class problem. This
  confirms that the remaining features carry genuine discriminative signal
  rather than class-encoded range artifacts.

  The 'from' column (node ID) was the strongest leakage source: a single
  depth-4 tree on node ID alone achieves acc=1.000. This is because each
  attack type was generated by a fixed set of nodes across all 10 simulation
  files. The model would learn node identity, not attack behaviour.

  TRAIN/TEST SPLIT RATIONALE
  --------------------------
  Files 0–7 are used for training (~8.2M packets, 80% of simulations).
  Files 8–9 are held out for testing (~2.0M packets, 20% of simulations).
  This file-based split is used instead of random cross-validation because
  the same physical node appears in all 10 files. Random folds would allow
  node behavioral fingerprints to leak across train/test boundaries.
""")


# ============================================================
# STEP — FILE-BASED TRAIN/TEST SPLIT
# ============================================================

def split_and_save(df_raw, df_clean):
    log("\n" + "=" * 70)
    log("TRAIN/TEST SPLIT — FILES 0-7 TRAIN, FILES 8-9 TEST")
    log("=" * 70)

    df_clean            = df_clean.copy()
    df_clean['file_id'] = df_raw['file_id']

    train = df_clean[df_clean['file_id'].isin(range(8))].drop(columns=['file_id'])
    test  = df_clean[df_clean['file_id'].isin([8, 9])].drop(columns=['file_id'])

    log(f"  Train (files 0-7): {len(train):,} rows")
    log(f"  Test  (files 8-9): {len(test):,} rows")

    log(f"\n  Train label distribution:")
    for cls, cnt in train[LABEL_COL].value_counts().items():
        log(f"    {str(cls).ljust(12)}: {cnt:,}")

    log(f"\n  Test label distribution:")
    for cls, cnt in test[LABEL_COL].value_counts().items():
        log(f"    {str(cls).ljust(12)}: {cnt:,}")

    log(f"\n  Class balance check (test/train ratio — expected ~0.25):")
    train_vc = train[LABEL_COL].value_counts()
    test_vc  = test[LABEL_COL].value_counts()
    for cls in sorted(train_vc.index):
        ratio = test_vc.get(cls, 0) / train_vc.get(cls, 1)
        log(f"    {str(cls).ljust(12)}: ratio = {ratio:.3f}")

    # Save
    log("\n  Saving outputs...")
    save_full = df_clean.drop(columns=['file_id'], errors='ignore')
    save_full.to_csv("iot_rpl_clean_full.csv", index=False)
    train.to_csv("iot_rpl_train.csv",           index=False)
    test.to_csv("iot_rpl_test.csv",             index=False)

    log(f"  Saved : iot_rpl_clean_full.csv  — {save_full.shape}")
    log(f"  Saved : iot_rpl_train.csv       — {train.shape}")
    log(f"  Saved : iot_rpl_test.csv        — {test.shape}")

    feature_cols = [c for c in save_full.columns if c != LABEL_COL]
    log(f"\n  Final features ({len(feature_cols)}):")
    for f in feature_cols:
        log(f"    {f}")

    return train, test


# ============================================================
# SAVE AUDIT REPORT TO FILE
# ============================================================

def save_audit_report():
    report_path = "leakage_audit_report.txt"
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(_audit_log))
    log(f"\n  Saved : {report_path}  (full audit log for methodology section)")


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    total_start = time.time()

    log("=" * 70)
    log("IoT-RPL 2021 — LEAKAGE AUDIT + CLEANING PIPELINE (FINAL)")
    log("=" * 70)

    # --------------------------------------------------------
    # STEP 1 — Load raw data
    # --------------------------------------------------------
    df_raw = step1_load()

    # --------------------------------------------------------
    # PHASE 0 — Leakage audit on raw data (proof for paper)
    # --------------------------------------------------------
    raw_flags, raw_leaky, raw_col_results = phase0_leakage_audit(df_raw)

    # --------------------------------------------------------
    # PHASE 1 — Clean the dataset
    # --------------------------------------------------------
    df = phase1_drop_bad_columns(df_raw)
    df = phase1_coerce_and_clean(df)

    # --------------------------------------------------------
    # PHASE 2 — Post-cleaning sanity checks
    # --------------------------------------------------------
    _, leaky2, _, _ = phase2_sanity_check_1(df)
    df              = phase2_drop_remaining(df, leaky2)
    clean_flags, clean_col_results = phase2_final_sanity_check(df)

    # --------------------------------------------------------
    # BEFORE / AFTER REPORT
    # --------------------------------------------------------
    all_dropped = (
        ID_LEAKY_COLS
        + HEX_COLS
        + leaky2
        + ['file_id']
    )
    all_dropped = list(dict.fromkeys(all_dropped))  # deduplicate, preserve order

    print_before_after_report(
        raw_flags,   raw_col_results,
        clean_flags, clean_col_results,
        all_dropped
    )

    # --------------------------------------------------------
    # SPLIT + SAVE
    # --------------------------------------------------------
    train, test = split_and_save(df_raw, df)

    # --------------------------------------------------------
    # SAVE FULL AUDIT LOG
    # --------------------------------------------------------
    save_audit_report()

    elapsed = time.time() - total_start
    log(f"\n{'=' * 70}")
    log(f"  Pipeline complete — total time: {elapsed/60:.1f} min")
    log(f"{'=' * 70}")

IoT-RPL 2021 — LEAKAGE AUDIT + CLEANING PIPELINE (FINAL)

STEP 1 — LOADING ALL 10 RAW FILES
  File 0 : 1,048,575 packets  (header from file)
  File 1 : 1,018,083 packets
  File 2 : 1,029,330 packets
  File 3 : 1,033,457 packets
  File 4 : 1,004,792 packets
  File 5 : 1,030,516 packets
  File 6 : 1,020,944 packets
  File 7 : 1,021,439 packets
  File 8 : 1,028,643 packets
  File 9 : 1,006,406 packets

  Total packets  : 10,242,185
  Total columns  : 24
  Load time      : 16.9s

  Label distribution (packet-level):
    Flooding    : 3,579,148
    Normal      : 2,143,817
    Blackhole   : 1,811,684
    Version     : 1,435,204
    Rank        : 1,272,332

PHASE 0 — LEAKAGE AUDIT: RAW UNCLEANED DATASET

  This phase documents the leakage present in the raw IoT-RPL 2021
  dataset BEFORE any cleaning. The findings here constitute the
  methodological justification for the cleaning steps that follow.

  Red flags to watch for (per feedback Appendix A):
    (a) Any column with constant value per